[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/20_clustering/02_choosing_k_and_failure_cases.ipynb)

# Bài 02 — Chọn số cụm K và khi nào K-Means thất bại

**🎯 Mục tiêu**
- Chọn $K$: elbow, **silhouette** (cài from scratch), Calinski-Harabasz, Davies-Bouldin — và giới hạn của chúng.
- Đánh giá khi có nhãn tham chiếu: ARI, NMI.
- Giả định ngầm của K-Means: cụm **lồi, đẳng hướng, kích thước tương tự** — 5 failure case.
- Template model cho K-Means.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/20_clustering

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import (adjusted_rand_score, calinski_harabasz_score, davies_bouldin_score, normalized_mutual_info_score,
                             silhouette_samples, silhouette_score)
from sklearn.preprocessing import StandardScaler

from ml_course.datasets import make_customer_segments
from ml_course.visualization import PALETTE, set_style

set_style()
cust = make_customer_segments(n=1500, seed=2)
features = ["annual_income_musd", "spending_score", "age", "visits_per_month"]
Xc = StandardScaler().fit_transform(cust[features])

## 1. Elbow: inertia theo K

Inertia **luôn giảm** khi tăng $K$ ($K = n$ → inertia = 0) → không chọn $K$ có inertia nhỏ nhất, mà tìm "khuỷu tay" nơi lợi ích giảm hẳn.

In [ ]:
Ks = range(1, 11)
inertia = [KMeans(k, n_init=10, random_state=0).fit(Xc).inertia_ for k in Ks]
plt.plot(Ks, inertia, "o-")
plt.xlabel("K")
plt.ylabel("inertia")
plt.title("Elbow — 'khuỷu tay' thường không rõ ràng trên dữ liệu thật")
plt.show()

## 2. Silhouette

Với điểm $i$: $a(i)$ = khoảng cách trung bình tới các điểm **cùng cụm**; $b(i)$ = khoảng cách trung bình nhỏ nhất tới các điểm của **một cụm khác**.

$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))} \in [-1, 1]$$

$s \approx 1$: điểm nằm sâu trong cụm của nó; $s \approx 0$: ở ranh giới; $s < 0$: có lẽ bị gán nhầm. Silhouette score = trung bình $s(i)$.

In [ ]:
def silhouette_scratch(X, labels):
    D = np.sqrt(((X[:, None, :] - X[None, :, :]) ** 2).sum(-1))
    s = np.zeros(len(X))
    for i in range(len(X)):
        same = labels == labels[i]
        same[i] = False
        if not same.any():
            continue
        a = D[i, same].mean()
        b = min(D[i, labels == c].mean() for c in np.unique(labels) if c != labels[i])
        s[i] = (b - a) / max(a, b)
    return s.mean()


sub = np.random.default_rng(0).choice(len(Xc), 500, replace=False)
lab5 = KMeans(5, n_init=10, random_state=0).fit_predict(Xc[sub])
print(f"silhouette (scratch) = {silhouette_scratch(Xc[sub], lab5):.4f} | sklearn = {silhouette_score(Xc[sub], lab5):.4f}")

In [ ]:
rows = []
for k in range(2, 11):
    lab = KMeans(k, n_init=10, random_state=0).fit_predict(Xc)
    rows.append({"K": k, "silhouette ↑": silhouette_score(Xc, lab), "Calinski-Harabasz ↑": calinski_harabasz_score(Xc, lab),
                 "Davies-Bouldin ↓": davies_bouldin_score(Xc, lab), "ARI (so với nhãn thật) ↑": adjusted_rand_score(cust.true_segment, lab)})
scores = pd.DataFrame(rows).set_index("K")
scores.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, col in zip(axes, ["silhouette ↑", "Calinski-Harabasz ↑", "Davies-Bouldin ↓"]):
    ax.plot(scores.index, scores[col], "o-")
    ax.axvline(5, color="gray", ls=":", label="K thật = 5")
    ax.set_title(col)
    ax.set_xlabel("K")
axes[0].legend()
plt.show()

Các chỉ số **nội tại** (không cần nhãn) không phải lúc nào cũng đồng ý với nhau, và không phải lúc nào cũng chỉ đúng "K thật" — chúng đo một khái niệm "cụm tốt" cụ thể (gọn, tách biệt). **Chọn K cuối cùng thường dựa vào mục đích sử dụng**: marketing có thể chăm sóc được bao nhiêu phân khúc? các cụm có diễn giải được không? có ổn định khi chạy lại trên dữ liệu khác không?

### Biểu đồ silhouette từng cụm

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, k in zip(axes, [3, 5]):
    lab = KMeans(k, n_init=10, random_state=0).fit_predict(Xc)
    s = silhouette_samples(Xc, lab)
    y0 = 0
    for c in range(k):
        sc = np.sort(s[lab == c])
        ax.fill_betweenx(np.arange(y0, y0 + len(sc)), 0, sc, color=PALETTE[c], alpha=0.7)
        y0 += len(sc) + 20
    ax.axvline(s.mean(), color="k", ls="--")
    ax.set_title(f"K = {k}: silhouette TB = {s.mean():.3f}")
    ax.set_xlabel("s(i)")
plt.show()

## 3. Đánh giá ngoài (khi có nhãn tham chiếu)

- **ARI** (Adjusted Rand Index): đồng thuận giữa hai cách chia, hiệu chỉnh cho ngẫu nhiên (0 = ngẫu nhiên, 1 = trùng khớp), **bất biến với tên cụm**.
- **NMI**: thông tin tương hỗ chuẩn hoá giữa hai cách chia.

Dùng để kiểm tra thuật toán trên dữ liệu có nhãn — trong thực tế thường **không có** nhãn (nếu có thì đã làm phân loại).

In [ ]:
lab = KMeans(5, n_init=10, random_state=0).fit_predict(Xc)
perm = (lab + 2) % 5                                                 # đổi tên cụm
print(f"ARI = {adjusted_rand_score(cust.true_segment, lab):.3f} | sau khi đổi tên cụm = {adjusted_rand_score(cust.true_segment, perm):.3f} | "
      f"NMI = {normalized_mutual_info_score(cust.true_segment, lab):.3f}")

## 4. Khi nào K-Means thất bại?

K-Means gán mỗi điểm về tâm gần nhất → ranh giới là **đường trung trực** giữa các tâm (Voronoi) → ngầm giả định cụm **lồi, tròn (đẳng hướng), kích thước và phương sai tương tự**.

In [ ]:
rng = np.random.default_rng(1)
cases = {}
Xm, ym = make_moons(400, noise=0.06, random_state=0)
cases["(a) hình không lồi"] = (Xm, ym, 2)
Xa, ya = make_blobs(600, centers=3, random_state=170)
cases["(b) dị hướng (kéo dài)"] = (Xa @ np.array([[0.6, -0.64], [-0.41, 0.85]]), ya, 3)
Xv, yv = make_blobs(600, centers=3, cluster_std=[0.4, 2.5, 0.4], random_state=8)
cases["(c) phương sai khác nhau"] = (Xv, yv, 3)
Xs = np.vstack([rng.normal([0, 0], 0.5, (500, 2)), rng.normal([3, 0], 0.5, (25, 2))])
cases["(d) kích thước rất khác nhau"] = (Xs, np.r_[np.zeros(500), np.ones(25)], 2)
Xo = np.vstack([rng.normal([0, 0], 0.5, (200, 2)), rng.normal([4, 0], 0.5, (200, 2)), [[20, 20], [21, 19]]])
cases["(e) outlier xa"] = (Xo, np.r_[np.zeros(200), np.ones(200), [2, 2]], 2)

fig, axes = plt.subplots(1, 5, figsize=(26, 4.6))
for ax, (name, (Xk, yk, k)) in zip(axes, cases.items()):
    lab = KMeans(k, n_init=10, random_state=0).fit_predict(Xk)
    ax.scatter(*Xk.T, c=[PALETTE[l] for l in lab], s=8)
    ax.set_title(f"{name}\nARI = {adjusted_rand_score(yk, lab):.2f}", fontsize=10)
plt.show()

| Failure | Khắc phục |
|---|---|
| (a) cụm không lồi (mặt trăng, vòng) | DBSCAN/HDBSCAN (bài 04), Spectral clustering, single linkage |
| (b) cụm dị hướng | chuẩn hoá/whitening, **Gaussian Mixture** với hiệp phương sai đầy đủ (chương 23) |
| (c) phương sai khác nhau | Gaussian Mixture |
| (d) kích thước rất khác nhau | GMM, DBSCAN; K-Means có xu hướng chia cụm lớn |
| (e) outlier | K-Medoids (tâm là điểm thật, dùng khoảng cách L1), loại outlier trước, DBSCAN (gán nhiễu) |

## 5. Template model: K-Means

| Phần | K-Means |
|---|---|
| Problem | phân cụm (không giám sát), lượng tử hoá vector, khởi tạo cho GMM |
| Intuition | mỗi cụm đại diện bởi một tâm; điểm thuộc tâm gần nhất |
| Geometry | ranh giới Voronoi (tuyến tính từng khúc), cụm lồi |
| Probability | trường hợp giới hạn của GMM với phương sai chung → 0 ("hard" EM) |
| Objective | $\min \sum_i \lVert\mathbf x_i - \boldsymbol\mu_{c_i}\rVert^2$ |
| Optimization | Lloyd (coordinate descent) — cực tiểu địa phương; k-means++, `n_init` |
| Assumptions | cụm lồi, đẳng hướng, kích thước/phương sai tương tự; K biết trước |
| Complexity | $O(n K d)$ mỗi vòng; Mini-Batch cho dữ liệu lớn |
| Hyperparameters | `n_clusters`, `init`, `n_init`, `max_iter` |
| Preprocessing | **scaling bắt buộc**; xử lý outlier; giảm chiều nếu nhiều chiều |
| Strengths | đơn giản, nhanh, dễ diễn giải (tâm), mở rộng tốt |
| Weaknesses | phải chọn K, nhạy outlier & scaling & khởi tạo, chỉ cụm lồi |
| Interpretation | profile tâm cụm (trung bình feature), kích thước cụm |

## 🧠 Kiểm tra nhanh

1. Vì sao không chọn K theo inertia nhỏ nhất?
2. Một điểm có $a(i) = 2$, $b(i) = 5$. $s(i)$ = ? Nếu $a(i) = 5$, $b(i) = 2$?
3. ARI có thay đổi khi đổi tên các cụm không?
4. Dữ liệu có hai cụm hình vòng tròn đồng tâm. K-Means có tách được không? Dùng gì?

## 📌 Tóm tắt

- Elbow/silhouette/CH/DB gợi ý K; quyết định cuối dựa vào mục đích, diễn giải, độ ổn định.
- ARI/NMI khi có nhãn tham chiếu.
- K-Means giả định cụm lồi, đẳng hướng, kích thước tương tự — thất bại với hình dạng phức tạp, phương sai khác nhau, outlier.

## 📚 Tài liệu tham khảo

- Rousseeuw (1987) — *Silhouettes: a graphical aid to the interpretation and validation of cluster analysis*
- Hubert & Arabie (1985) — *Comparing partitions* (ARI)
- scikit-learn User Guide — Clustering performance evaluation